# Projets 
 - Multi-query 
 - Routing vector stores ou bdd images 
 - Query construction pour rechercher images
 - Indexing Classique
 - CRAG

In [ ]:
import requests
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool,ToolRuntime
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from dataclasses import dataclass
load_dotenv()

True

In [ ]:
# 1. On initialise le modèle
model = init_chat_model("gpt-4.1-mini", model_provider="openai",temperature=0.5)

#### Indexing

In [ ]:
# Diviser le texte en paragraphes 
# 1. Lire le contenu du fichier
with open("documentOriginal.txt", "r", encoding="utf-8") as fichier:
    texte = fichier.read()

# 2. Faire des paragraphes
texte_modifie = texte.replace("Article détaillé", "<paragraphe>")
paragraphe = "<paragraphe>"

# 4. Réécrire le texte modifié dans le fichier (ou un nouveau fichier)
with open("document_modifie.txt", "w", encoding="utf-8") as fichier:
    fichier.write(paragraphe+texte_modifie+paragraphe)

In [ ]:
tableau_paragraphes = texte_modifie.split("<paragraphe>")
print(f"Nombre de paragraphes : {len(tableau_paragraphes)}")

Nombre de paragraphes : 31


In [ ]:
# base de docnnées vectoriel 
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import create_retriever_tool

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)
vectorstore = FAISS.from_texts(
    texts=tableau_paragraphes, 
    embedding=embeddings
)


C:\Users\pierr\AppData\Local\Temp\ipykernel_48884\2501699412.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
docs = vectorstore.similarity_search("qu' a fait le general de Gaulle", k=3)
print(docs)
liste_ids = [doc.id for doc in docs]
print(liste_ids)

[Document(id='e47b3b7c-1ebe-4533-9114-6b45363b3fbd', metadata={}, page_content=" : Histoire de la France pendant la Seconde Guerre mondiale.\n Soldats allemands défilant devant l'Arc de Triomphe à Paris, 14 juin 1940. \nAprès avoir déclaré la guerre le 3 septembre à l'Allemagne à la suite de son entrée en Pologne, la France tente avec le Royaume-Uni de secourir la Norvège victime d'un même assaut allemand ; sans succès probant. Cette « drôle de guerre » où il ne se passe pas grand-chose sur le front prend fin le 10 mai 1940 avec une offensive éclair (blitzkrieg) de l'Axe qui conquiert la France (directement la partie nord) en six semaines. Pourtant, Philippe Pétain avait fait construire la ligne Maginot le long de la frontière franco-allemande (il aurait voulu la construire également le long de la frontière belge mais le roi Léopold III voulait conserver la neutralité de son pays). Les Allemands sont passés par la Belgique et la forêt des Ardennes, et grâce à l'aide des chars et de l'a

#### Multi-query

In [ ]:
# 1 On crée l'agent en associant le modèle et les outils
agent_multiquery = create_agent(
    model=model,
    system_prompt="Tu es assistant expert reformulation. Tu propose 5 reformulations questions pour la question posée par l'utilisateur. Ces questions doivent être le plus precis possible pour alimenter une recherche sur une base de données vectorielle",
)
# 2 On envoie un message simple
response = agent_multiquery.invoke({
    'messages':[
        {"role": "user", "content": "Qu'a fait le generale de gaulle pendant la seconde guerre mondiale ?"}
    ]
})
# 3 On affiche la réponse
print (response['messages'][-1].content)

1. Quelles actions militaires et politiques le général de Gaulle a-t-il entreprises durant la Seconde Guerre mondiale ?
2. Quel rôle a joué le général de Gaulle dans la résistance française entre 1939 et 1945 ?
3. Quelles sont les principales contributions du général de Gaulle à la libération de la France pendant la Seconde Guerre mondiale ?
4. Comment le général de Gaulle a-t-il organisé et dirigé la France libre pendant la Seconde Guerre mondiale ?
5. Quelles décisions stratégiques le général de Gaulle a-t-il prises lors de la Seconde Guerre mondiale pour soutenir les Alliés ?


In [ ]:
from collections import Counter

# On récupère les k ids les plus fréquents dans la liste des ids
def getUniqueIDs(liste_docs,k=3):
    compteur = Counter(liste_ids)
    ids_classes = compteur.most_common()
    result = []
    for i in range(k):
        id_ = ids_classes[i][0]
        result.append((id_))
    return result

# Recherche de documents similaires pour chaque question générée par l'agent
docs = []
ids =[]
for questions in response['messages'][-1].content.split("\n"):
    docs.extend(vectorstore.similarity_search(questions, k=3))

# On affiche les ids des documents trouvés
print([doc.id for doc in docs])

# On récupère les k ids les plus fréquents dans la liste des ids et on retourne leurs contenus
ids = getUniqueIDs([doc.id for doc in docs],k=3)
for id in ids:
    print([doc.page_content for doc in docs if doc.id == id])
    

['e47b3b7c-1ebe-4533-9114-6b45363b3fbd', '427501f6-6c84-4379-a491-1cd35fb49980', 'cfb47e2f-a8d5-4480-bf9f-7b41415cbc09', 'e47b3b7c-1ebe-4533-9114-6b45363b3fbd', '427501f6-6c84-4379-a491-1cd35fb49980', 'cfb47e2f-a8d5-4480-bf9f-7b41415cbc09', 'e47b3b7c-1ebe-4533-9114-6b45363b3fbd', '427501f6-6c84-4379-a491-1cd35fb49980', 'df5a1ba1-cde5-48c4-8773-54627f039107', 'e47b3b7c-1ebe-4533-9114-6b45363b3fbd', '427501f6-6c84-4379-a491-1cd35fb49980', 'cfb47e2f-a8d5-4480-bf9f-7b41415cbc09', 'e47b3b7c-1ebe-4533-9114-6b45363b3fbd', '427501f6-6c84-4379-a491-1cd35fb49980', 'cfb47e2f-a8d5-4480-bf9f-7b41415cbc09']
[" : Histoire de la France pendant la Seconde Guerre mondiale.\n Soldats allemands défilant devant l'Arc de Triomphe à Paris, 14 juin 1940. \nAprès avoir déclaré la guerre le 3 septembre à l'Allemagne à la suite de son entrée en Pologne, la France tente avec le Royaume-Uni de secourir la Norvège victime d'un même assaut allemand ; sans succès probant. Cette « drôle de guerre » où il ne se passe p

In [ ]:
# on recapitule 
@tool('retriever', description="A tool to retrieve documents from a question")
def retriever(question:str, k:int=3):
    # 1 On crée l'agent en associant le modèle et les outils
    agent_multiquery = create_agent(
        model=model,
        system_prompt="Tu es assistant expert reformulation. Tu propose 5 reformulations questions pour la question posée par l'utilisateur. Ces questions doivent être le plus precis possible pour alimenter une recherche sur une base de données vectorielle",
    )
    # 2 On envoie un message simple
    response = agent_multiquery.invoke({
        'messages':[
            {"role": "user", "content": question}
        ]
    })
    # 3 On affiche la réponse
    # print (response['messages'][-1].content)

    # Recherche de documents similaires pour chaque question générée par l'agent
    docs = []
    ids =[]
    for questions in response['messages'][-1].content.split("\n"):
        docs.extend(vectorstore.similarity_search(questions, k=k))

    # On récupère les k ids les plus fréquents dans la liste des ids et on retourne leurs contenus
    ids = getUniqueIDs([doc.id for doc in docs],k=k)
    result = []
    for id in ids:
        result.extend([doc.page_content for doc in docs if doc.id == id])
        # print([doc.page_content for doc in docs if doc.id == id])
    return result

# retriever_tool = create_retriever_tool(retriever,
#     name="faiss_retriever",
#     description="A tool to retrieve documents from a question")

# 1 On crée l'agent en associant le modèle et les outils
agent_reponse = create_agent(
    tools=[retriever],
    model=model,
    system_prompt="Tu es un assistant expert en histoire tu reponds aux questions en utilisant les informations du retriever tool",
)
# 2 On envoie un message simple
response = agent_reponse.invoke({
    'messages':[
        {"role": "user", "content": "Qu'a fait le generale de gaulle pendant la seconde guerre mondiale ?"}
    ]
})
# 3 On affiche la réponse
print (response['messages'][-1].content)

Pendant la Seconde Guerre mondiale, le général Charles de Gaulle a joué un rôle crucial dans la résistance française contre l'occupation nazie. Après la défaite de la France en juin 1940 et la signature de l'armistice par le maréchal Pétain, de Gaulle s'est opposé à cet armistice. Le 18 juin 1940, depuis Londres, il a lancé son célèbre appel à la résistance via la radio BBC, incitant les Français à poursuivre le combat aux côtés des Alliés britanniques contre les nazis.

De Gaulle est devenu le chef de la France libre, un mouvement qui regroupait les Forces françaises libres. Il a réussi à rallier plusieurs colonies françaises, notamment en Afrique-Équatoriale française et au Cameroun, formant ainsi l'Afrique française libre. La France libre a continué à combattre sur plusieurs fronts, notamment en Libye, en Égypte, en Tunisie et en Italie.

Après le débarquement des Alliés en Normandie en 1944, de Gaulle a affirmé que la France combattait aux côtés des Alliés comme un allié à part ent